In [2]:
import pandas as pd
import numpy as np

# read raw data（if you have df，start from "data fill"）
FILE_PATH = r"./SOFR_SWAP.wide.xls"

df = pd.read_csv(FILE_PATH, sep="\t")

for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------ data fill：11Y、13Y、14Y ------------

def fill_missing_years(row):
    """
    fill 11Y, 13Y, 14Y
    rules：
      11Y : using 10Y and 12Y data linear interpolation
      13Y : using 12Y and 15Y, weight 1/3
      14Y : using 12Y and 15Y,weight 2/3
    only fill when there are data besides；otherwise keep NaN
    """
    # for missing data 11Y
    if "11Y" in row.index and "10Y" in row.index and "12Y" in row.index:
        if pd.isna(row["11Y"]) and row[["10Y", "12Y"]].notna().all():
            row["11Y"] = row["10Y"] + (row["12Y"] - row["10Y"]) * (11 - 10) / (12 - 10)

    # for missing data 13Y
    if "13Y" in row.index and "12Y" in row.index and "15Y" in row.index:
        if pd.isna(row["13Y"]) and row[["12Y", "15Y"]].notna().all():
            row["13Y"] = row["12Y"] + (row["15Y"] - row["12Y"]) * (13 - 12) / (15 - 12)

    # for missing data 14Y
    if "14Y" in row.index and "12Y" in row.index and "15Y" in row.index:
        if pd.isna(row["14Y"]) and row[["12Y", "15Y"]].notna().all():
            row["14Y"] = row["12Y"] + (row["15Y"] - row["12Y"]) * (14 - 12) / (15 - 12)

    return row

df_filled = df.apply(fill_missing_years, axis=1)


df_filled.to_csv("SOFR_SWAP.wide.filled.xls", sep="\t", index=False)
